# 🛡️ CodeSentinel AI — Train Model 1: Defect & Smell Sentinel
### Fine-Tuning GraphCodeBERT on Microsoft CodeXGLUE (Free T4 GPU in Google Colab)

This notebook trains **Model 1 (The Defect & Anti-Pattern Sentinel)** on the **CodeXGLUE Defect Detection** benchmark (~27,000 real-world functions).

* **Hardware**: Free Google Colab T4 GPU (16 GB VRAM)
* **Base Model**: `microsoft/graphcodebert-base`
* **Output**: Production-ready quantized model pushed directly to your free Hugging Face account.

## 1. Verify GPU Hardware
Ensure you have selected **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi

: 

## 2. Install Required Dependencies

In [ ]:
!pip install --quiet transformers datasets accelerate scikit-learn huggingface_hub

## 3. Load CodeXGLUE Defect Detection Dataset
Loads the published benchmark dataset (~21.8k training, 2.7k validation, 2.7k test functions).

In [ ]:
from datasets import load_dataset

print("📥 Loading CodeXGLUE Defect Detection dataset...")
dataset = load_dataset("google/code_x_glue_cc_defect_detection")
print(dataset)
print("\nSample record:")
print(dataset["train"][0])

## 4. Setup GraphCodeBERT Tokenizer & Preprocessing

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_ID = "microsoft/graphcodebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

def preprocess_function(examples):
    tokenized = tokenizer(examples["func"], truncation=True, max_length=384)
    tokenized["labels"] = [int(t) for t in examples["target"]]
    return tokenized

print("🔄 Tokenizing dataset...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 5. Define Model Architecture & Evaluation Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: "Clean", 1: "Defective"},
    label2id={"Clean": 0, "Defective": 1},
    problem_type="single_label_classification",
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## 6. Train with Hugging Face Trainer (Mixed Precision fp16)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=0.1,
    fp16=True,  # Fast 16-bit mixed precision on T4 GPU
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=1,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Starting Training on T4 GPU (Estimated time: ~30-35 mins)...")
trainer.train()

## 7. Evaluate on the Final Test Split

In [ ]:
print("📊 Evaluating on CodeXGLUE Test Set (2,732 functions)...")
test_results = trainer.evaluate(tokenized_dataset["test"])
print("=" * 50)
print(f"Test Accuracy:  {test_results['eval_accuracy'] * 100:.2f}%")
print(f"Test Precision: {test_results['eval_precision'] * 100:.2f}%")
print(f"Test Recall:    {test_results['eval_recall'] * 100:.2f}%")
print(f"Test F1-Score:  {test_results['eval_f1'] * 100:.2f}%")
print("=" * 50)

## 8. Export & Push to Hugging Face Model Hub (Free)
Enter your Hugging Face Access Token (from https://huggingface.co/settings/tokens with Write permission).

In [ ]:
from huggingface_hub import notebook_login

print("🔑 Sign in to Hugging Face:")
notebook_login()

In [ ]:
# Set your HF username and target repo name
HF_USERNAME = "wraithklu"  # Replace with your Hugging Face username
REPO_NAME = f"{HF_USERNAME}/codesentinel-defect-sentinel"

print(f"📤 Pushing best model to: https://huggingface.co/{REPO_NAME}")
trainer.save_model("./codesentinel-model")
tokenizer.save_pretrained("./codesentinel-model")

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"🎉 SUCCESS! Model is live at https://huggingface.co/{REPO_NAME}")